In [ ]:
%pip install -r requirements.txt

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats.mstats import winsorize


In [ ]:
kagglepath =  "../../dataset/kgdataset.csv"
surveypath = "../../dataset/survey_data.xlsx"
df = pd.read_csv(kagglepath)
df2 = pd.read_excel(surveypath)
df2.drop(df2.columns[0],axis=1, inplace=True)
df2.columns = df.columns
combined = pd.concat([df, df2], ignore_index=True)
combined


In [ ]:
dfclean = pd.DataFrame(combined)
dfclean["Feel Rested"] = dfclean["Feel Rested"].replace("ไม่สดชื่น", "No").replace("สดชื่น", "Yes")
dfclean["Use Before Sleep"] = dfclean["Use Before Sleep"].replace("ไม่ใช่", "No").replace("ใช่", "Yes")
dfclean["Anxiety/Low Mood"] = dfclean["Anxiety/Low Mood"].replace("ไม่หงุดหงิด/ไม่วิตกกังวล", "No").replace("หงุดหงิด/วิตกกังวล", "Yes")
dfclean["Wellness Apps"] = dfclean["Wellness Apps"].replace("ไม่ใช่", "No").replace("ใช่", "Yes")
dfclean["Sleep Quality"] = dfclean["Sleep Quality"].replace("ไม่ดี", "Bad").replace("ดี", "Good")
dfclean["Screen Time Affects Sleep?"] = dfclean["Screen Time Affects Sleep?"].replace("ไม่แน่ใจ", "Not Sure").replace("ใช่", "Yes").replace("ไม่ใช่", "No")


In [ ]:
#check for null values and duplicates
print(dfclean.isnull().sum())
duplicates = dfclean[dfclean.duplicated()]
print("Duplicate Rows:")
print(duplicates)
dfclean = dfclean.drop_duplicates()
print("DataFrame after removing duplicates:")
print(dfclean)

In [ ]:
#clean age
def extract_number(value):
    if "-" in str(value):
        value = re.sub(r'[^\d\-\.]', '', value)
        values = str(value).split("-")
        return f"{values[0]}-{values[1]}"
    elif isinstance(value, str):
        result = pd.Series(value).str.extract('(\d+)') 
        return result[0].iloc[0] if not result.empty else None

    else:
        return str(value)      
dfclean["Age"] = dfclean["Age"].apply(extract_number)

#clean sleep hours by mean
def clean_range_with_mean(value):
    if isinstance(value, str):
        if '-' in value:
            try:
                low, high = map(float, value .split('-'))
                return (low + high) / 2
            except ValueError:
                return value
        else:
            try:
                return float(value)
            except ValueError:
                return value
    else:
        return value
dfclean["Sleep Hours"] = dfclean["Sleep Hours"].apply(extract_number)
dfclean["Sleep Hours"] = dfclean["Sleep Hours"].apply(clean_range_with_mean)

#Clean Daily Screen Time
dfclean["Daily Screen Time"] = dfclean["Daily Screen Time"].apply(extract_number)
dfclean["Daily Screen Time"] = dfclean["Daily Screen Time"].apply(clean_range_with_mean)
print(dfclean["Sleep Hours"].isnull().sum())
print(dfclean["Age"].isnull().sum())
print(dfclean["Daily Screen Time"].isnull().sum())


In [ ]:
dfclean.info()


In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(dfclean['Screen Time Affects Sleep?'], kde=False)
plt.title('Distribution of Screen Time Affects Sleep?')

In [ ]:
dfclean["Age"] = pd.to_numeric(dfclean["Age"], errors='coerce')
numeric_cols = dfclean.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    sns.histplot(dfclean[col], kde=True)
    plt.title(f'Distribution of {col}')

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(12, 6))
    sns.boxplot(x=dfclean[col])

In [ ]:
dfclean['Daily Screen Time'] = winsorize(dfclean['Daily Screen Time'],(0.1, 0.1))
dfclean['Age'] = winsorize(dfclean['Age'],(0.1, 0.1))

for col in numeric_cols:
    plt.figure(figsize=(12, 6))
    sns.boxplot(x=dfclean[col])
    plt.title(f'Box Plot of {col}')
    plt.show()

In [ ]:
sns.pairplot(dfclean[numeric_cols])
plt.show()

In [ ]:
sns.stripplot(data=dfclean, x="Daily Screen Time", y="Sleep Hours", jitter=True, alpha=0.5)

In [ ]:
cat_cols = dfclean.select_dtypes(include=['object']).columns
enumerated_cat_cols = list(enumerate(cat_cols))
fig, axes = plt.subplots(nrows=1, ncols=6, figsize=(20, 4))
for i, col in enumerated_cat_cols:
    sns.countplot(data=dfclean, x=col, ax=axes[i])

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=6, figsize=(20, 4))
sns.boxplot(data=df, x="Use Before Sleep", y="Sleep Hours", ax=axes[0])
sns.boxplot(data=df, x="Sleep Quality", y="Sleep Hours", ax=axes[1])
sns.boxplot(data=df, x="Anxiety/Low Mood", y="Sleep Hours", ax=axes[2])


In [ ]:
sns.boxplot(data=df, x="Feel Rested", y="Sleep Hours")


In [ ]:
numeric_data = dfclean.select_dtypes(include=['number'])
correlation_matrix = numeric_data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)

